**Setup Widgets & Parameters**

In [0]:
#1. Setup Widgets (UI Inputs)

dbutils.widgets.dropdown("pipeline_stage","bronze", ["bronze","silver","gold"])
dbutils.widgets.text("process_date","2026-01-01")

#2. Get Values
current_stage = dbutils.widgets.get("pipeline_stage")
process_date = dbutils.widgets.get("process_date")
#current_stage = ""
print(f" JOB CONFIGURATION:")
print(f"   • Current Stage: {current_stage.upper()}")
print(f"   • Processing Date: {process_date}")


#Define global paths

base_path = "/Volumes/workspace/ecommerce/ecommerce_data"
paths = {
    "raw": f"{base_path}",
    "bronze": f"{base_path}/bronze_events",
    "silver": f"{base_path}/silver_events", 
    "gold": f"{base_path}/gold_product"
    }

**Logic Routing (The Controller)**

In [0]:
from pyspark.sql.functions import col,current_timestamp,to_date,when,countDistinct,sum,lit

#Bronze Layer

def run_bronze():
    print("Starting Bronze Ingestion")
    df_raw = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true")\
                .csv(f"{paths['raw']}/2019-*.csv")
    
    df_bronze = df_raw.withColumn("ingestion_ts", current_timestamp())

    df_bronze.write.mode("overwrite")\
                    .format("delta")\
                    .save(paths['bronze'])

    print(f"Finished Bronze Ingestion at {paths['bronze']}")

#Silver Layer

def run_silver():
    print("Starting Silver Ingestion")
    df_bronze = spark.read.format("delta").load(paths['bronze'])

    df_silver = df_bronze.filter((col("price").cast("float") > 0) & (col("price").cast("float") < 50000))\
                        .dropDuplicates(["user_session","event_time","product_id"])\
                        .withColumn("event_date", to_date(col("event_time"))) \
                        .withColumn("price_tier", when(col("price").cast("float") > 50, "Inexpensive").otherwise("Luxury"))

    df_silver.write.format("delta").mode("overwrite").save(paths['silver'])

    print(f"Finished Silver Ingestion at {paths['silver']}")

def run_gold():
    print("Starting Gold Ingestion")
    df_silver = spark.read.format("delta").load(paths['silver'])

    df_gold = df_silver.groupBy("product_id","category_code","brand")\
                        .agg(
                            # Count unique users who VIEWED
                            countDistinct(when(col("event_type") == "view", col("user_id"))).alias("unique_views"),
                            # Count unique users who PURCHASED
                            countDistinct(when(col("event_type") == "purchase", col("user_id"))).alias("unique_purchases"),
                            # Total Revenue
                            sum(when(col("event_type") == "purchase", col("price"))).alias("total_revenue")
                        )

    df_gold_final = df_gold.withColumn(
        "conversion_rate_pct", 
        (col("unique_purchases") / (col("unique_views") + 1)) * 100
    ).fillna(0)

    df_gold_final.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(paths['gold'])

    print(f"Finished Gold Ingestion at {paths['gold']}")

**Execution Block**

In [0]:
current_stage = "gold"

if current_stage == "bronze":
    run_bronze()
elif current_stage == "silver":
    run_silver()
elif current_stage == "gold":
    run_gold()
else:
    raise ValueError(f"Unknown stage: {current_stage}")